# AgentMeter — Phase 5 Pilot (Google Colab, free T4)

Per-agent resource benchmarking of one open-source LLM in a **minimal linear**
Perceive → Reason → Decide → Act network-threat-detection pipeline.

Dataset: `data/cicids_pilot.csv` (real CIC-IDS flows, 78 features). Taxonomy is
**5 classes**: Brute Force, Volumetric DDoS, Port Scanning, DoS Hulk, Benign.

**Tahap 1 guardrails (unchanged):**
- Pipeline is the *test subject* only — minimal, generic, linear. No governance,
  correlation, alerting, or retry/self-correction loops.
- **Sequential** execution only (never parallel — it contaminates readings).
- **Mode A**: model pulled from the HF Hub and run **in-process** on the GPU, so
  per-agent VRAM is measurable via `torch.cuda` / `pynvml`.
- All config in `config.yaml` / the pilot config. Nothing hard-coded.
- GPU required: if `torch.cuda` is unavailable the pilot **STOPS** (no CPU fallback).

### ⚠️ Quantization notice (declare in your thesis)
A 7–8B model does **not** fit in fp16 on a 15 GB T4. This pilot uses **4-bit NF4**
quantization (bitsandbytes). The VRAM / latency / token numbers are therefore for
the **quantized** model, and any model-vs-model comparison must use the **same**
quantization to stay fair. This is a methodological choice you must state.

### Cost
Colab free tier has no \$ cost, but sessions are time-limited and the GPU can be
reclaimed. Save `pilot.json` as soon as the run finishes.

## 1. Confirm you have a GPU runtime
Runtime → Change runtime type → Hardware accelerator = **T4 GPU**. Then run:

In [ ]:
!nvidia-smi -L || echo 'NO GPU: set Runtime -> Change runtime type -> T4 GPU'

## 2. Get the AgentMeter code
Clones the repo and checks out the working branch. If the repo is **private**,
paste a GitHub token when prompted (input is hidden). If it is public, just press
Enter to skip.

In [ ]:
import os, getpass, subprocess
REPO = 'https://github.com/ismazahin/AgentMeter'
BRANCH = 'claude/agentmeter-phases-0-5-3ftnmx'
gh = getpass.getpass('GitHub token (press Enter if repo is public): ').strip()
url = REPO.replace('https://', f'https://{gh}@') if gh else REPO
if not os.path.isdir('AgentMeter'):
    subprocess.run(['git', 'clone', '--branch', BRANCH, url + '.git', 'AgentMeter'], check=True)
os.chdir('AgentMeter')
subprocess.run(['git', 'checkout', BRANCH], check=True)
print('cwd:', os.getcwd())
print('branch:', subprocess.run(['git','rev-parse','--abbrev-ref','HEAD'],capture_output=True,text=True).stdout.strip())

## 3. Install dependencies
Colab already ships a CUDA build of `torch` — we do **not** reinstall it (that
would risk breaking CUDA). We add the CPU-set deps plus the GPU/HF extras and
`bitsandbytes` for 4-bit.

In [ ]:
# CPU-set deps (langgraph, pandas, numpy, scipy, pyyaml) — no torch here
!pip install -q -r requirements.txt
# GPU / HF extras (torch already present on Colab)
!pip install -q transformers accelerate huggingface_hub sentencepiece pynvml bitsandbytes
import torch; print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No CUDA GPU — switch runtime to T4 before continuing.'

## 4. Authenticate with your Hugging Face token (secure)
Entered via `getpass` — **not** hardcoded, not printed, not saved to the notebook.
You must have accepted the model's gated licence on its Hub page first.

Alternatively use a Colab Secret named `HF_TOKEN` (🔑 panel) — the cell picks it up.

In [ ]:
import os, getpass
tok = ''
try:
    from google.colab import userdata
    tok = userdata.get('HF_TOKEN') or ''
except Exception:
    pass
if not tok:
    tok = getpass.getpass('Enter your HF_TOKEN (hidden): ').strip()
assert tok, 'HF_TOKEN is required for gated models.'
os.environ['HF_TOKEN'] = tok
print('HF_TOKEN set (', len(tok), 'chars ). Not displayed.')

## 5. Choose the models to benchmark
Both models run **sequentially** (never in parallel) at the same **4-bit NF4**
quantization, same config, same scenarios — so the comparison is fair. Accept
each model's gated licence on its HF page first, or the download will 401.

In [ ]:
CONFIG = 'configs/pilot_colab_t4.yaml'
MODELS = [
    'mistralai/Mistral-7B-Instruct-v0.3',
    'meta-llama/Meta-Llama-3-8B-Instruct',
]
print('Will benchmark, one after another:')
for m in MODELS: print('  -', m)

## 6. Run BOTH models (sequential, clean VRAM isolation)
For each model: free the GPU → load (4-bit) → run all scenarios → save
`results/pilot_<model>.json` → **fully unload + free VRAM** → next model.
The first model is completely freed before the second loads, so each model's
VRAM readings are clean. First load of each also downloads its weights
(~4–5 GB in 4-bit; one-time, excluded from per-agent timings). Also writes
`results/pilot_combined.json`.

In [ ]:
from agentmeter.pilot import run_pilot_models
results = run_pilot_models(models=MODELS, config_path=CONFIG, n=10)

## 7. Download the result files
Send these back for the side-by-side comparison demo: the two per-model
files and the combined file.

In [ ]:
import glob
from google.colab import files
for f in sorted(glob.glob('results/pilot_*.json')):
    print('downloading', f)
    try:
        files.download(f)
    except Exception as e:
        print('  download manually from the Files panel:', e)

## 8. Per-model report (optional)
Render the single-model HTML report for each model (pure stdlib, no extra
installs). The side-by-side Mistral-vs-Llama comparison is a separate local
Gradio demo that reads these saved JSON files.

In [ ]:
import glob
from IPython.display import HTML, display
for f in sorted(glob.glob('results/pilot_*.json')):
    if f.endswith('pilot_combined.json'): continue
    out = f.replace('.json', '_report.html')
    !python scripts/report.py "{f}" -o "{out}"
    display(HTML(open(out).read()))

## 9. Done — free the GPU
Runtime → Disconnect and delete runtime, so the free GPU is released for your
next session. Send me the two per-model JSON files (and pilot_combined.json)
and I'll build the side-by-side comparison demo.